# Notebook 1: Análisis Exploratorio de Datos (EDA) y Data Augmentation

**Autores:** Javier Arroyo | Julia Cano | Paula Durá  
**Asignatura:** Procesamiento de Imágenes  

---

## Objetivo

Este notebook constituye la **fase inicial del pipeline** de procesamiento de imágenes. Antes de entrenar cualquier modelo, es fundamental comprender las características del dataset: distribución de clases, resoluciones, propiedades de color y posibles anomalías.

Posteriormente, aplicamos técnicas de **Data Augmentation** para ampliar el dataset de ~150 a 500 imágenes por clase, lo cual es esencial para evitar el *overfitting* en modelos de Deep Learning.

### Contenido
| Sección | Descripción |
|---------|-------------|
| 1.1 – 1.2 | Configuración e indexación del dataset |
| 1.3 – 1.5 | Análisis de distribución, integridad y visualización |
| 1.6 – 1.8 | Análisis de resoluciones, color y outliers |
| 2.1 – 2.4 | Data Augmentation: estrategia, generación y verificación |

---
## 1.1 Configuración inicial

Importamos las librerías necesarias y fijamos una **semilla global** (`SEED = 42`) para garantizar la reproducibilidad de todos los experimentos.

In [ ]:
import os
import glob
import random
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from PIL import Image, ImageStat, ImageFile, ImageFilter, ImageEnhance
ImageFile.LOAD_TRUNCATED_IMAGES = True

import cv2
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

In [ ]:
DATA_DIR = Path("dataset")
classes = sorted([p.name for p in DATA_DIR.iterdir() if p.is_dir()])
print("Clases encontradas:", classes)

---
## 1.2 Indexación del dataset

Recorremos la estructura de carpetas para construir un **DataFrame** con la información de cada imagen:
- **Ruta** completa del archivo
- **Clase** (nombre de la carpeta contenedora)
- **Nombre** del archivo y extensión

Este DataFrame será la estructura central para todo el análisis posterior.

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

rows = []
for cls in classes:
    for p in (DATA_DIR / cls).rglob("*"):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            rows.append({"path": str(p), "class": cls, "filename": p.name, "ext": p.suffix.lower()})

df = pd.DataFrame(rows)
print(f"Total imágenes: {len(df)}")
df.head()

---
## 1.3 Distribución de clases

Un dataset **balanceado** es crucial para evitar que los modelos favorezcan una clase sobre otra. Verificamos cuántas imágenes hay por categoría.

In [ ]:
counts = df["class"].value_counts().sort_index()
display(counts)

plt.figure()
counts.plot(kind="bar", color=["#2ecc71","#3498db","#e74c3c","#f39c12","#9b59b6"])
plt.title("Número de imágenes por clase")
plt.ylabel("count")
plt.tight_layout()
plt.show()

**Interpretación:** El dataset está bien balanceado con aproximadamente 150 imágenes por clase. Sin embargo, este tamaño es **insuficiente para entrenar redes neuronales profundas** de forma efectiva (donde típicamente se necesitan miles de ejemplos por clase). Por ello, en la sección 2 aplicaremos técnicas de Data Augmentation para expandir el dataset.

---
## 1.4 Integridad del dataset

Antes de cualquier procesamiento, verificamos que **todas las imágenes se pueden abrir correctamente** y revisamos sus modos de color (RGB, RGBA, L, etc.). Las imágenes corruptas se excluyen del análisis.

In [ ]:
bad_files = []
modes = []

for p in df["path"]:
    try:
        with Image.open(p) as im:
            modes.append(im.mode)
    except Exception as e:
        bad_files.append({"path": p, "error": str(e)})

bad_df = pd.DataFrame(bad_files)
print(f"Archivos corruptos: {len(bad_df)}")
if len(bad_df) > 0:
    display(bad_df.head())

mode_counts = pd.Series(modes).value_counts()
display(mode_counts)

# Si hay corruptas, las quitamos
if len(bad_df) > 0:
    df = df[~df["path"].isin(bad_df["path"])].reset_index(drop=True)

print(f"Dataset limpio: {df.shape[0]} imágenes")

---
## 1.5 Muestra visual por clase

Visualizamos ejemplos representativos de cada categoría para:
- Verificar que las **etiquetas son correctas** (no hay imágenes mal clasificadas)
- Detectar posible **ruido visual** (imágenes con marcas de agua, collages, etc.)
- Entender la **variabilidad intra-clase** (diversidad de contenido dentro de cada categoría)

In [ ]:
def show_grid(paths, title, n=12, cols=4):
    sample = paths[:n]
    rows = int(np.ceil(len(sample) / cols))
    plt.figure(figsize=(12, 3*rows))
    for i, p in enumerate(sample):
        im = Image.open(p).convert("RGB")
        plt.subplot(rows, cols, i+1)
        plt.imshow(im)
        plt.axis("off")
        plt.title(Path(p).name[:25], fontsize=8)
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

for cls in classes:
    cls_paths = df[df["class"]==cls]["path"].sample(min(12, (df["class"]==cls).sum()), random_state=SEED).tolist()
    show_grid(cls_paths, title=f"Ejemplos - {cls}", n=12)

---
## 1.6 Análisis de resoluciones y aspect ratio

Las redes neuronales requieren imágenes de **tamaño fijo**. Necesitamos entender la distribución de resoluciones para elegir la estrategia de redimensionado más adecuada (*resize*, *crop* o *padding*).

In [ ]:
meta = []
for p in df["path"]:
    with Image.open(p) as im:
        w, h = im.size
        meta.append({"path": p, "width": w, "height": h, "aspect_ratio": w / h})

meta_df = pd.DataFrame(meta)
df = df.merge(meta_df, on="path", how="left")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df["width"], bins=30, color="#3498db", alpha=0.7)
axes[0].set_title("Distribución de anchos (px)")
axes[0].set_xlabel("width")

axes[1].hist(df["height"], bins=30, color="#e74c3c", alpha=0.7)
axes[1].set_title("Distribución de altos (px)")
axes[1].set_xlabel("height")

for cls in classes:
    d = df[df["class"]==cls]
    axes[2].scatter(d["width"], d["height"], s=10, alpha=0.5, label=cls)
axes[2].set_title("Width vs Height")
axes[2].set_xlabel("width")
axes[2].set_ylabel("height")
axes[2].legend(fontsize=7)

plt.tight_layout()
plt.show()

---
## 1.7 Análisis de color y distribución de píxeles

Estudiamos las propiedades cromáticas del dataset para entender si existen **patrones de color discriminativos** entre clases. Esto es relevante porque:
- Puede indicar qué *features* serán útiles para la clasificación
- Ayuda a decidir si normalizar los canales por separado o conjuntamente

In [ ]:
def rgb_brightness_stats(path):
    im = Image.open(path).convert("RGB")
    arr = np.asarray(im).astype(np.float32)
    flat = arr.reshape(-1, 3)
    mean_rgb = flat.mean(axis=0)
    std_rgb  = flat.std(axis=0)
    gray = 0.299*arr[...,0] + 0.587*arr[...,1] + 0.114*arr[...,2]
    return mean_rgb, std_rgb, gray.mean(), gray.std()

stats = []
for p, cls in zip(df["path"], df["class"]):
    mean_rgb, std_rgb, b_mean, b_std = rgb_brightness_stats(p)
    stats.append({
        "path": p, "class": cls,
        "mean_r": mean_rgb[0], "mean_g": mean_rgb[1], "mean_b": mean_rgb[2],
        "std_r": std_rgb[0], "std_g": std_rgb[1], "std_b": std_rgb[2],
        "brightness_mean": b_mean, "brightness_std": b_std,
    })

stats_df = pd.DataFrame(stats)
df = df.merge(stats_df, on=["path","class"], how="left")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df.boxplot(column="brightness_mean", by="class", ax=axes[0], rot=45)
axes[0].set_title("Brillo medio por clase")
axes[0].set_ylabel("brightness")

means = df.groupby("class")[["mean_r","mean_g","mean_b"]].mean().sort_index()
means.plot(kind="bar", ax=axes[1], color=["red","green","blue"], alpha=0.7)
axes[1].set_title("Media RGB por clase")
axes[1].set_ylabel("mean channel value")

plt.suptitle("")
plt.tight_layout()
plt.show()

### Histogramas RGB agregados por clase

Para profundizar en el análisis cromático, visualizamos la distribución conjunta de los canales R, G y B. Esto revela las **"firmas de color"** características de cada categoría.

In [ ]:
def plot_rgb_hist_for_class(cls, n=30, bins=64):
    sample_paths = df[df["class"]==cls]["path"].sample(min(n, (df["class"]==cls).sum()), random_state=SEED).tolist()
    all_pixels = []
    for p in sample_paths:
        arr = np.asarray(Image.open(p).convert("RGB"))
        all_pixels.append(arr.reshape(-1,3))
    all_pixels = np.vstack(all_pixels)

    plt.figure(figsize=(8, 3))
    plt.hist(all_pixels[:,0], bins=bins, alpha=0.5, color="red", label="R")
    plt.hist(all_pixels[:,1], bins=bins, alpha=0.5, color="green", label="G")
    plt.hist(all_pixels[:,2], bins=bins, alpha=0.5, color="blue", label="B")
    plt.title(f"Histograma RGB agregado - {cls}")
    plt.xlabel("pixel value")
    plt.ylabel("count")
    plt.legend()
    plt.tight_layout()
    plt.show()

for cls in classes:
    plot_rgb_hist_for_class(cls)

**Interpretación:** Las distribuciones de brillo y los histogramas RGB confirman que cada clase tiene una **firma cromática diferenciada**:
- **Playa** → dominancia de azules (cielo/mar) y amarillos (arena)
- **Naturaleza** → predominancia del canal verde (vegetación)
- **Ciudad** → distribución uniforme tendiente a grises (edificios, asfalto)
- **Comida** → tonos cálidos (rojos, naranjas, amarillos)
- **Animales** → alta variabilidad cromática

Estas diferencias sugieren que los modelos de clasificación podrán aprovechar la información de color como *feature* discriminativa.

---
## 1.8 Detección de outliers

Identificamos imágenes potencialmente problemáticas que podrían afectar al entrenamiento:
- **Resolución muy baja** (<128px): podrían perder detalle al redimensionar
- **Aspect ratio extremo** (<0.5 o >2.0): distorsión al hacer *resize* cuadrado
- **Brillo anómalo**: imágenes sobreexpuestas o subexpuestas

In [ ]:
small = df[(df["width"] < 128) | (df["height"] < 128)]
weird_ar = df[(df["aspect_ratio"] < 0.5) | (df["aspect_ratio"] > 2.0)]
dark = df[df["brightness_mean"] < 40]
bright = df[df["brightness_mean"] > 220]

print(f"Pequeñas (<128px):       {len(small)}")
print(f"Aspect ratio extremo:    {len(weird_ar)}")
print(f"Muy oscuras (<40):       {len(dark)}")
print(f"Muy claras (>220):       {len(bright)}")

if len(dark) > 0:
    show_grid(dark["path"].head(12).tolist(), "Ejemplos de imágenes oscuras", n=12)
if len(weird_ar) > 0:
    show_grid(weird_ar["path"].head(12).tolist(), "Ejemplos de aspect ratio extremo", n=12)

**Conclusión:** No se detectan outliers problemáticos que requieran eliminación. El dataset es de buena calidad y se puede usar directamente para entrenamiento.

---

# 2. Data Augmentation

## Motivación

Con ~150 imágenes por clase, nuestro dataset es **insuficiente para Deep Learning**, donde los modelos suelen necesitar miles de ejemplos para generalizar correctamente. El Data Augmentation es una técnica estándar que:

1. **Aumenta el tamaño efectivo del dataset** → reduce el *overfitting*
2. **Introduce variabilidad realista** → mejora la generalización a nuevos datos
3. **Preserva el balance entre clases** → garantiza representación equitativa

## Estrategia

Aplicamos **9 tipos de transformaciones** que preservan la semántica de la imagen (una playa sigue siendo una playa tras el *flip* o cambio de brillo):

| Transformación | Descripción | Justificación |
|---|---|---|
| `hflip` | Volteo horizontal | Las escenas naturales son simétricas |
| `rotate` | Rotación ±15° | Simula variación de encuadre/ángulo |
| `brightness` | Ajuste de brillo (0.7–1.4) | Simula distintas condiciones de luz |
| `contrast` | Ajuste de contraste (0.7–1.4) | Variación de condiciones atmosféricas |
| `color` | Saturación (0.7–1.3) | Simula diferentes cámaras |
| `crop` | Recorte aleatorio (75-95%) | Zoom parcial, centra la atención |
| `blur` | Desenfoque gaussiano leve | Simula movimiento o desenfoque |
| `noise` | Ruido gaussiano (σ=5-15) | Robustez ante cámaras de baja calidad |
| `combined` | Composición de 2-3 anteriores | Máxima variabilidad |

**Objetivo:** Alcanzar **500 imágenes por clase** (de ~150 a ~500 = ×3.3 de expansión).

In [ ]:
# ── Configuración de Data Augmentation ──────────────────────────────────────

AUGMENTED_DIR = Path("dataset_augmented")
TARGET_PER_CLASS = 500   # Objetivo: 500 imágenes por clase (de ~150 originales)
IMG_SIZE_AUG = (224, 224)  # Tamaño estándar para los modelos

def augment_image(img, aug_type):
    """Aplica una transformación de augmentación a una imagen PIL."""
    if aug_type == "hflip":
        return img.transpose(Image.FLIP_LEFT_RIGHT)
    elif aug_type == "rotate":
        angle = random.uniform(-15, 15)
        return img.rotate(angle, resample=Image.BILINEAR, fillcolor=(128,128,128))
    elif aug_type == "brightness":
        factor = random.uniform(0.7, 1.4)
        return ImageEnhance.Brightness(img).enhance(factor)
    elif aug_type == "contrast":
        factor = random.uniform(0.7, 1.4)
        return ImageEnhance.Contrast(img).enhance(factor)
    elif aug_type == "color":
        factor = random.uniform(0.7, 1.3)
        return ImageEnhance.Color(img).enhance(factor)
    elif aug_type == "crop":
        w, h = img.size
        crop_frac = random.uniform(0.75, 0.95)
        new_w, new_h = int(w * crop_frac), int(h * crop_frac)
        left = random.randint(0, w - new_w)
        top = random.randint(0, h - new_h)
        return img.crop((left, top, left + new_w, top + new_h)).resize((w, h), Image.BILINEAR)
    elif aug_type == "blur":
        radius = random.uniform(0.5, 1.5)
        return img.filter(ImageFilter.GaussianBlur(radius=radius))
    elif aug_type == "noise":
        arr = np.array(img).astype(np.float32)
        noise = np.random.normal(0, random.uniform(5, 15), arr.shape)
        arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
        return Image.fromarray(arr)
    elif aug_type == "combined":
        # Combinación de 2-3 augmentaciones
        augs = random.sample(["hflip", "rotate", "brightness", "contrast", "color", "crop"], k=random.randint(2, 3))
        for a in augs:
            img = augment_image(img, a)
        return img
    return img

AUG_TYPES = ["hflip", "rotate", "brightness", "contrast", "color", "crop", "blur", "noise", "combined"]

print(f"Tipos de augmentation disponibles: {len(AUG_TYPES)}")
print(f"Objetivo: {TARGET_PER_CLASS} imágenes por clase")

---
### 2.1 Visualización de transformaciones

Antes de generar el dataset completo, verificamos visualmente que las transformaciones producen resultados **realistas y coherentes**. Cada fila muestra una imagen original junto con todas sus posibles augmentaciones.

In [ ]:
# Tomar una imagen de ejemplo de cada clase
fig, axes = plt.subplots(len(classes), len(AUG_TYPES) + 1, figsize=(22, 3 * len(classes)))

for i, cls in enumerate(classes):
    sample_path = df[df["class"] == cls]["path"].iloc[0]
    orig = Image.open(sample_path).convert("RGB").resize(IMG_SIZE_AUG)
    
    axes[i, 0].imshow(orig)
    axes[i, 0].set_title("Original", fontsize=7)
    axes[i, 0].axis("off")
    if i == 0:
        axes[i, 0].set_title("Original", fontsize=8, fontweight="bold")
    axes[i, 0].set_ylabel(cls, rotation=0, labelpad=50, fontsize=9, va="center")
    
    for j, aug_type in enumerate(AUG_TYPES):
        augmented = augment_image(orig.copy(), aug_type)
        axes[i, j+1].imshow(augmented)
        axes[i, j+1].axis("off")
        if i == 0:
            axes[i, j+1].set_title(aug_type, fontsize=8, fontweight="bold")

plt.suptitle("Ejemplos de Data Augmentation por clase y tipo", fontsize=14)
plt.tight_layout()
plt.show()

---
### 2.2 Generación del dataset aumentado

Proceso de generación:
1. **Copia de originales:** Todas las imágenes originales se redimensionan a 224×224 y se incluyen en el dataset aumentado
2. **Augmentación aleatoria:** Se generan imágenes adicionales aplicando transformaciones aleatorias hasta alcanzar el objetivo de 500 por clase
3. **Guardado:** Las imágenes se almacenan en `dataset_augmented/` con la misma estructura de carpetas

In [ ]:
import shutil

# Crear estructura de carpetas
for cls in classes:
    (AUGMENTED_DIR / cls).mkdir(parents=True, exist_ok=True)

aug_stats = {}

for cls in classes:
    cls_df = df[df["class"] == cls]
    original_paths = cls_df["path"].tolist()
    n_originals = len(original_paths)
    n_needed = max(0, TARGET_PER_CLASS - n_originals)
    
    count = 0
    
    # 1) Copiar originales (resize a tamaño estándar)
    for p in original_paths:
        img = Image.open(p).convert("RGB").resize(IMG_SIZE_AUG)
        out_path = AUGMENTED_DIR / cls / f"orig_{count:04d}.jpg"
        img.save(out_path, "JPEG", quality=95)
        count += 1
    
    # 2) Generar augmentaciones adicionales
    for i in range(n_needed):
        src_path = random.choice(original_paths)
        img = Image.open(src_path).convert("RGB").resize(IMG_SIZE_AUG)
        aug_type = random.choice(AUG_TYPES)
        img_aug = augment_image(img, aug_type)
        out_path = AUGMENTED_DIR / cls / f"aug_{count:04d}.jpg"
        img_aug.save(out_path, "JPEG", quality=95)
        count += 1
    
    aug_stats[cls] = {"originals": n_originals, "augmented": n_needed, "total": count}
    print(f"{cls}: {n_originals} originales + {n_needed} augmentadas = {count} total")

print(f"\nDataset aumentado guardado en: {AUGMENTED_DIR}")

---
### 2.3 Verificación del dataset aumentado

Comparamos la distribución antes y después de la augmentación, y mostramos ejemplos del dataset resultante para confirmar la calidad visual.

In [ ]:
# Verificar distribución del dataset aumentado
aug_rows = []
for cls in classes:
    for p in (AUGMENTED_DIR / cls).rglob("*.jpg"):
        aug_rows.append({"path": str(p), "class": cls})

aug_df = pd.DataFrame(aug_rows)
aug_counts = aug_df["class"].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Antes
counts.plot(kind="bar", ax=axes[0], color=["#2ecc71","#3498db","#e74c3c","#f39c12","#9b59b6"])
axes[0].set_title(f"ANTES: Dataset original ({len(df)} imgs)")
axes[0].set_ylabel("count")
axes[0].tick_params(axis='x', rotation=45)

# Después
aug_counts.plot(kind="bar", ax=axes[1], color=["#2ecc71","#3498db","#e74c3c","#f39c12","#9b59b6"])
axes[1].set_title(f"DESPUÉS: Dataset aumentado ({len(aug_df)} imgs)")
axes[1].set_ylabel("count")
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Mostrar ejemplos del dataset aumentado
for cls in classes:
    aug_paths = aug_df[aug_df["class"] == cls]["path"].sample(min(8, len(aug_df[aug_df["class"]==cls])), random_state=SEED).tolist()
    show_grid(aug_paths, title=f"Dataset aumentado - {cls}", n=8, cols=4)

---
### 2.4 Resumen y conclusiones

| Aspecto | Resultado |
|---------|-----------|
| **Clases** | 5 (Animales, Ciudad, Comida, Naturaleza, Playa) |
| **Imágenes originales** | ~150 por clase (~750 total) |
| **Imágenes tras augmentation** | 500 por clase (2500 total) |
| **Factor de expansión** | ×3.3 |
| **Balance** | Perfectamente equilibrado |
| **Archivos corruptos** | Ninguno detectado |
| **Outliers** | No significativos |
| **Firmas de color** | Diferenciadas por clase → útil para clasificación |

### Próximos pasos

El dataset aumentado (`dataset_augmented/`) se utilizará en los notebooks siguientes:
- **Notebook 02** → Clasificación de imágenes (SVM, CNN, Transfer Learning)
- **Notebook 03** → Detección de objetos (YOLOv8)
- **Notebook 04** → Generación de imágenes (cDCGAN, Stable Diffusion)

In [ ]:
# Guardar el dataframe del EDA para los otros notebooks
df.to_csv("eda_dataframe.csv", index=False)
print("DataFrame del EDA guardado en eda_dataframe.csv")
print(f"Dataset original: {DATA_DIR}")
print(f"Dataset aumentado: {AUGMENTED_DIR}")